In [0]:
%sql
CREATE OR REPLACE TABLE supplychainiq.ops.lost_sales_truth AS
SELECT to_date(date) AS date, store_id, item_id, demand, sold, lost
FROM read_files('/Volumes/supplychainiq/ops/ground_truth/lost_sales.parquet')
WHERE to_date(date) <= '2015-08-20';          -- only the period we've ingested

SELECT COUNT(*) AS rows, SUM(lost) AS true_lost_units FROM supplychainiq.ops.lost_sales_truth;

In [0]:
%sql
CREATE OR REPLACE TABLE supplychainiq.gold.fact_item_day
CLUSTER BY (snapshot_date, store_id) AS
WITH sales AS (
  SELECT event_date AS d, store_id, item_id,
         SUM(units) AS units, SUM(units * unit_price) AS revenue
  FROM supplychainiq.silver.pos_events GROUP BY ALL),
price AS (
  SELECT store_id, item_id, SUM(units * unit_price) / SUM(units) AS avg_price
  FROM supplychainiq.silver.pos_events GROUP BY ALL),
base AS (
  SELECT i.snapshot_date, i.store_id, i.item_id,
         split(i.item_id, '_')[0]                                     AS cat_id,
         concat_ws('_', split(i.item_id, '_')[0], split(i.item_id, '_')[1]) AS dept_id,
         i.on_hand,
         COALESCE(s.units, 0)   AS units_sold,
         COALESCE(s.revenue, 0) AS revenue,
         i.on_hand = 0          AS is_stockout          -- end-of-day snapshot: shelf ran empty
  FROM supplychainiq.silver.inventory i
  LEFT JOIN sales s ON s.d = i.snapshot_date AND s.store_id = i.store_id AND s.item_id = i.item_id),
w AS (
  SELECT *,
         AVG(CASE WHEN NOT is_stockout THEN units_sold END)
           OVER (PARTITION BY store_id, item_id ORDER BY snapshot_date
                 ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS baseline_28d,
         VAR_SAMP(CASE WHEN NOT is_stockout THEN units_sold END)
           OVER (PARTITION BY store_id, item_id ORDER BY snapshot_date
                 ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS var_28d
  FROM base)
SELECT w.*, p.avg_price,
       CASE WHEN is_stockout THEN GREATEST(COALESCE(baseline_28d, 0) - units_sold, 0) ELSE 0 END
         AS est_lost_units,
       CASE WHEN is_stockout THEN GREATEST(COALESCE(baseline_28d, 0) - units_sold, 0) ELSE 0 END
         * COALESCE(p.avg_price, 0) AS est_lost_revenue
FROM w LEFT JOIN price p USING (store_id, item_id);

In [0]:
%sql
CREATE OR REPLACE TABLE supplychainiq.gold.store_day_kpis AS
WITH launched AS (            -- an item counts only from its first sale in that store
  SELECT store_id, item_id, MIN(snapshot_date) AS first_sale
  FROM supplychainiq.gold.fact_item_day WHERE units_sold > 0 GROUP BY ALL)
SELECT f.snapshot_date AS date, f.store_id,
       f.snapshot_date < '2014-06-20'                                     AS is_warmup,  -- no 28-day baseline yet
       SUM(f.units_sold)                                                   AS units_sold,
       ROUND(SUM(f.revenue), 2)                                            AS revenue,
       COUNT_IF(f.snapshot_date >= l.first_sale)                           AS active_items,
       COUNT_IF(f.is_stockout AND f.snapshot_date >= l.first_sale)         AS stockout_items,
       ROUND(1 - COUNT_IF(f.is_stockout AND f.snapshot_date >= l.first_sale)
                 / COUNT_IF(f.snapshot_date >= l.first_sale), 4)           AS in_stock_rate,
       ROUND(SUM(f.est_lost_units), 1)                                     AS est_lost_units,
       ROUND(SUM(f.est_lost_revenue), 2)                                   AS est_lost_revenue,
       ROUND(SUM(f.est_lost_units) / (SUM(f.units_sold) + SUM(f.est_lost_units)), 4) AS est_lost_rate
FROM supplychainiq.gold.fact_item_day f
LEFT JOIN launched l USING (store_id, item_id)
GROUP BY ALL;

SELECT store_id, ROUND(AVG(in_stock_rate), 4) AS avg_in_stock, ROUND(SUM(revenue)) AS revenue,
       ROUND(SUM(est_lost_revenue)) AS est_lost_revenue, ROUND(AVG(est_lost_rate), 4) AS avg_lost_rate
FROM supplychainiq.gold.store_day_kpis
WHERE NOT is_warmup
GROUP BY store_id ORDER BY est_lost_revenue DESC;

In [0]:
%sql
WITH est AS (SELECT store_id, date_trunc('month', snapshot_date) AS m, SUM(est_lost_units) AS est
             FROM supplychainiq.gold.fact_item_day WHERE snapshot_date >= '2014-06-20' GROUP BY ALL),
     tru AS (SELECT store_id, date_trunc('month', date) AS m, SUM(lost) AS actual
             FROM supplychainiq.ops.lost_sales_truth WHERE date >= '2014-06-20' GROUP BY ALL),
     j   AS (SELECT COALESCE(e.store_id, t.store_id) AS store_id, COALESCE(e.m, t.m) AS m,
                    COALESCE(est, 0) AS est, COALESCE(actual, 0) AS actual
             FROM est e FULL JOIN tru t ON e.store_id = t.store_id AND e.m = t.m)
SELECT ROUND(SUM(est))                              AS est_lost_total,
       SUM(actual)                                  AS true_lost_total,
       ROUND(SUM(est) / SUM(actual), 3)             AS est_to_true_ratio,
       ROUND(SUM(ABS(est - actual)) / SUM(actual), 3) AS wape_store_month
FROM j;

In [0]:
%sql
WITH f AS (SELECT snapshot_date AS d, store_id, item_id, is_stockout, baseline_28d, units_sold, est_lost_units
           FROM supplychainiq.gold.fact_item_day WHERE snapshot_date >= '2014-06-20'),
     t AS (SELECT date AS d, store_id, item_id, lost
           FROM supplychainiq.ops.lost_sales_truth WHERE date >= '2014-06-20')
SELECT CASE WHEN f.d IS NULL                     THEN '0_missing_in_gold'
            WHEN NOT f.is_stockout                THEN '1_not_flagged_stockout'
            WHEN f.baseline_28d IS NULL           THEN '2_no_baseline'
            WHEN f.units_sold >= f.baseline_28d   THEN '3_sold_above_baseline'
            ELSE                                       '4_estimated_but_low' END AS bucket,
       COUNT(*)                          AS lost_days,
       SUM(t.lost)                       AS true_lost,
       ROUND(SUM(COALESCE(f.est_lost_units, 0))) AS our_estimate,
       ROUND(SUM(t.lost) / SUM(SUM(t.lost)) OVER (), 3) AS share_of_true_loss
FROM t LEFT JOIN f USING (d, store_id, item_id)
GROUP BY 1 ORDER BY 1;

In [0]:
%sql
SELECT COUNT(*) AS stockout_days_without_true_loss, ROUND(SUM(f.est_lost_units)) AS over_estimate_units
FROM supplychainiq.gold.fact_item_day f
LEFT ANTI JOIN supplychainiq.ops.lost_sales_truth t
  ON t.date = f.snapshot_date AND t.store_id = f.store_id AND t.item_id = f.item_id
WHERE f.is_stockout AND f.snapshot_date >= '2014-06-20';

In [0]:
import numpy as np, pandas as pd
from scipy import stats
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")
def expected_lost(mean: pd.Series, var: pd.Series, sold: pd.Series) -> pd.Series:
    """E[D - a | D >= a] for D ~ Poisson (var<=mean) or Negative Binomial (var>mean)."""
    m = mean.fillna(0).to_numpy(float)
    a = sold.fillna(0).to_numpy(float)
    v = var.to_numpy(float)
    v = np.where(np.isnan(v), m, v)
    out = np.zeros_like(m)
    nb = (m > 0) & (v > m * 1.01)
    po = (m > 0) & ~nb
    if po.any():                                   # k*p(k; lam) = lam * p(k-1; lam)
        lam, aa = m[po], a[po]
        num, den = stats.poisson.sf(aa - 2, lam), stats.poisson.sf(aa - 1, lam)
        out[po] = np.where(den > 1e-12, lam * num / np.maximum(den, 1e-12) - aa, 0)
    if nb.any():                                   # k*p(k; r,p) = mean * p(k-1; r+1,p)
        mu, aa = m[nb], a[nb]
        r = mu**2 / (v[nb] - mu); p = r / (r + mu)
        num, den = stats.nbinom.sf(aa - 2, r + 1, p), stats.nbinom.sf(aa - 1, r, p)
        out[nb] = np.where(den > 1e-12, mu * num / np.maximum(den, 1e-12) - aa, 0)
    return pd.Series(np.maximum(out, 0.0))

fact = spark.table("supplychainiq.gold.fact_item_day")
v2 = (fact.withColumn("est_lost_units_v2",
          F.when(F.col("is_stockout"),
                 expected_lost(F.col("baseline_28d"), F.col("var_28d"), F.col("units_sold")))
           .otherwise(F.lit(0.0)))
          .withColumn("est_lost_revenue_v2", F.col("est_lost_units_v2") * F.coalesce(F.col("avg_price"), F.lit(0))))
v2.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("supplychainiq.gold.fact_item_day_v2")
print("done:", spark.table("supplychainiq.gold.fact_item_day_v2").count(), "rows")

In [0]:
%sql
WITH est AS (SELECT store_id, date_trunc('month', snapshot_date) AS m,
                    SUM(est_lost_units) AS v1, SUM(est_lost_units_v2) AS v2
             FROM supplychainiq.gold.fact_item_day_v2 WHERE snapshot_date >= '2014-06-20' GROUP BY ALL),
     tru AS (SELECT store_id, date_trunc('month', date) AS m, SUM(lost) AS actual
             FROM supplychainiq.ops.lost_sales_truth WHERE date >= '2014-06-20' GROUP BY ALL),
     j   AS (SELECT COALESCE(v1,0) v1, COALESCE(v2,0) v2, COALESCE(actual,0) actual
             FROM est FULL JOIN tru USING (store_id, m))
SELECT 'v1 avg - sold'           AS method, ROUND(SUM(v1)) AS est_total, SUM(actual) AS true_total,
       ROUND(SUM(v1)/SUM(actual),3) AS ratio, ROUND(SUM(ABS(v1-actual))/SUM(actual),3) AS wape FROM j
UNION ALL
SELECT 'v2 conditional expectation', ROUND(SUM(v2)), SUM(actual),
       ROUND(SUM(v2)/SUM(actual),3), ROUND(SUM(ABS(v2-actual))/SUM(actual),3) FROM j;

In [0]:
from pyspark.sql import functions as F, Window

w = Window.partitionBy("store_id", "item_id").orderBy("snapshot_date").rowsBetween(-28, -1)
df = (spark.table("supplychainiq.gold.fact_item_day_v2")
        .withColumn("disp", F.when((F.col("baseline_28d") > 0) & (F.col("var_28d") > F.col("baseline_28d")),
                                   F.col("var_28d") / F.col("baseline_28d")).otherwise(F.lit(1.0)))  # keep each item's lumpiness
        .withColumn("mu", F.col("baseline_28d")))

tmp = ["supplychainiq.gold._em_a", "supplychainiq.gold._em_b"]          # alternate tables: never read & overwrite the same one
for i in range(3):
    df = (df.withColumn("imputed",                                       # E-step
                F.when(F.col("is_stockout"),
                       F.col("units_sold") + expected_lost(F.col("mu"), F.col("mu") * F.col("disp"), F.col("units_sold")))
                 .otherwise(F.col("units_sold").cast("double")))
            .withColumn("mu", F.coalesce(F.avg("imputed").over(w), F.col("baseline_28d"))))   # M-step: all days count now
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tmp[i % 2])
    df = spark.table(tmp[i % 2])                                         # materialise -> short lineage, stable on serverless
    print(f"EM iteration {i+1} done")

v3 = (df.withColumn("est_lost_units_v3",
            F.when(F.col("is_stockout"),
                   expected_lost(F.col("mu"), F.col("mu") * F.col("disp"), F.col("units_sold")))
             .otherwise(F.lit(0.0)))
        .withColumn("est_lost_revenue_v3", F.col("est_lost_units_v3") * F.coalesce(F.col("avg_price"), F.lit(0)))
        .drop("imputed"))
v3.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("supplychainiq.gold.fact_item_day_v3")
spark.sql("DROP TABLE IF EXISTS supplychainiq.gold._em_a")
spark.sql("DROP TABLE IF EXISTS supplychainiq.gold._em_b")
print("v3 rows:", spark.table("supplychainiq.gold.fact_item_day_v3").count())

In [0]:
%sql
WITH est AS (SELECT store_id, date_trunc('month', snapshot_date) AS m,
                    SUM(est_lost_units)    AS v1,
                    SUM(est_lost_units_v2) AS v2,
                    SUM(est_lost_units_v3) AS v3
             FROM supplychainiq.gold.fact_item_day_v3
             WHERE snapshot_date >= '2014-06-20' GROUP BY ALL),
     tru AS (SELECT store_id, date_trunc('month', date) AS m, SUM(lost) AS actual
             FROM supplychainiq.ops.lost_sales_truth
             WHERE date >= '2014-06-20' GROUP BY ALL),
     j   AS (SELECT COALESCE(v1,0) v1, COALESCE(v2,0) v2, COALESCE(v3,0) v3, COALESCE(actual,0) actual
             FROM est FULL JOIN tru USING (store_id, m))
SELECT 'v1 avg - sold' AS method, ROUND(SUM(v1)) AS est_total, SUM(actual) AS true_total,
       ROUND(SUM(v1)/SUM(actual),3) AS ratio, ROUND(SUM(ABS(v1-actual))/SUM(actual),3) AS wape FROM j
UNION ALL
SELECT 'v2 conditional expectation', ROUND(SUM(v2)), SUM(actual),
       ROUND(SUM(v2)/SUM(actual),3), ROUND(SUM(ABS(v2-actual))/SUM(actual),3) FROM j
UNION ALL
SELECT 'v3 EM censored demand', ROUND(SUM(v3)), SUM(actual),
       ROUND(SUM(v3)/SUM(actual),3), ROUND(SUM(ABS(v3-actual))/SUM(actual),3) FROM j;

In [0]:
%sql
WITH f AS (SELECT snapshot_date AS d, store_id, item_id, units_sold, est_lost_units_v3
           FROM supplychainiq.gold.fact_item_day_v3 WHERE is_stockout AND snapshot_date >= '2014-06-20'),
     t AS (SELECT date AS d, store_id, item_id, lost FROM supplychainiq.ops.lost_sales_truth WHERE date >= '2014-06-20')
SELECT CASE WHEN f.units_sold = 0 THEN 'empty_all_day' ELSE 'sold_out_during_day' END AS kind,
       COUNT(*) AS days, SUM(COALESCE(t.lost,0)) AS true_lost, ROUND(SUM(f.est_lost_units_v3)) AS v3_est,
       ROUND(SUM(f.est_lost_units_v3) / SUM(COALESCE(t.lost,0)), 3) AS ratio
FROM f LEFT JOIN t USING (d, store_id, item_id) GROUP BY 1;

In [0]:
%sql
CREATE OR REPLACE TABLE supplychainiq.gold.fact_item_day CLUSTER BY (snapshot_date, store_id) AS
SELECT snapshot_date, store_id, item_id, cat_id, dept_id, on_hand, units_sold,
       ROUND(revenue, 2)               AS revenue,
       is_stockout, avg_price,
       ROUND(mu, 3)                    AS demand_baseline_28d,   -- EM-corrected baseline
       ROUND(est_lost_units_v3, 3)     AS est_lost_units,        -- official estimator (v3, conservative)
       ROUND(est_lost_revenue_v3, 2)   AS est_lost_revenue
FROM supplychainiq.gold.fact_item_day_v3;

COMMENT ON TABLE supplychainiq.gold.fact_item_day IS
  'Store x item x day. est_lost_* uses EM censored-demand estimator (v3): validated lower bound, ~70% of true lost units, WAPE 0.30 at store-month.';